# Data Preprocessing — Flats
**Improvements:** Fixed `FutureWarning` from chained `fillna(inplace=True)`, added safe price parsing with length check, documented all steps.

In [ ]:
import numpy as np
import pandas as pd
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
df = pd.read_csv('flats.csv')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("Duplicates:", df.duplicated().sum())
print("Missing values:\n", df.isnull().sum())

In [ ]:
# Drop irrelevant columns
df.drop(columns=['link', 'property_id'], inplace=True)

# Rename area column
df.rename(columns={'area': 'price_per_sqft'}, inplace=True)

In [ ]:
# Clean society: remove star ratings, lowercase
df['society'] = df['society'].apply(
    lambda name: re.sub(r'\d+(\.\d+)?\s?\u2605', '', str(name)).strip().lower()
)
print(f"Unique societies after cleaning: {df['society'].nunique()}")

In [ ]:
# Remove 'Price on Request' rows
df = df[df['price'] != 'Price on Request'].copy()
print(f"Rows after removing 'Price on Request': {len(df)}")

In [ ]:
# Parse price: 'X Lac' -> X/100 Crores, 'X Crore' -> X Crores
def treat_price(val):
    parts = val.split(' ')
    # FIX: validate length before indexing to avoid IndexError
    if len(parts) < 2:
        return np.nan
    num = float(parts[0])
    unit = parts[1]
    if unit == 'Lac':
        return round(num / 100, 2)
    elif unit == 'Cr':
        return round(num, 2)
    return np.nan

df['price'] = df['price'].apply(treat_price)
df = df[df['price'].notna()].copy()

In [ ]:
# Parse price_per_sqft
df['price_per_sqft'] = (
    df['price_per_sqft']
    .str.split('/').str[0]
    .str.replace('\u20b9', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

In [ ]:
# Parse bedRoom
df['bedRoom'] = df['bedRoom'].str.split(' ').str[0]
df = df[df['bedRoom'] != 'nan'].copy()
df['bedRoom'] = df['bedRoom'].astype(int)
print(f"Rows after bedRoom cleanup: {len(df)}")

In [ ]:
# Parse bathroom
df['bathroom'] = df['bathroom'].str.split(' ').str[0].astype(int)

In [ ]:
# Parse balcony: 'No Balcony' -> '0'
df['balcony'] = df['balcony'].str.split(' ').str[0].str.replace('No', '0')

In [ ]:
# FIX: avoid FutureWarning — use direct assignment instead of fillna(inplace=True) on slice
df['additionalRoom'] = df['additionalRoom'].fillna('not available')
df['additionalRoom'] = df['additionalRoom'].str.lower()

In [ ]:
# Parse floorNum
def parse_floor(val):
    val = str(val).lower()
    if 'ground' in val: return 0
    if 'basement' in val or 'lower' in val: return -1
    match = re.search(r'\d+', val)
    return int(match.group()) if match else np.nan

df['floorNum'] = df['floorNum'].apply(parse_floor)

In [ ]:
# FIX: direct assignment instead of inplace on slice
df['facing'] = df['facing'].fillna('NA')

In [ ]:
# Derive built_up area from price and price_per_sqft
df['area'] = round((df['price'] * 10000000) / df['price_per_sqft'])

# Add property type tag
df['property_type'] = 'flat'

print(f"Final shape: {df.shape}")
df.info()

In [ ]:
df.to_csv('flats_cleaned.csv', index=False)
print("Saved flats_cleaned.csv")